# OneVoice V2 — data audit
Notebook chạy hai tầng: logical audit nhanh trước, physical audit 24.192 WAV sau. Physical audit trên Google Drive có thể rất lâu vì mỗi WAV là một file remote riêng; nó không chạy mặc định cho đến khi bạn chủ động bật công tắc.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
WORK_ROOT = MYDRIVE / 'OneVoice'
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'soundfile'], check=True)
DATASET_ROOT = MYDRIVE / 'onevoice_audio_v1'
MANIFEST = DATASET_ROOT / 'manifest.jsonl'
REPORT_ROOT = WORK_ROOT / 'reports/data_audit_v1'
print('Source:', REPO, '| Data:', DATASET_ROOT, '| Reports:', REPORT_ROOT)


In [ ]:
def run_streaming(command):
    print('>', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    print('Exit code:', returncode, flush=True)
    return returncode


## Bước 1 — Phục hồi manifest nếu thiếu

In [ ]:
if not MANIFEST.is_file():
    print('manifest.jsonl is missing; recovering deterministic fields from V1 filenames...', flush=True)
    code = run_streaming([sys.executable, 'scripts/recover_v1_manifest.py', '--dataset-root', str(DATASET_ROOT), '--metadata-csv', 'data/onevoice_construction_v2/utterances_all.csv', '--output', str(MANIFEST)])
    if code != 0: raise RuntimeError('Manifest recovery failed; inspect the log above')
else:
    print('Manifest already exists:', MANIFEST)


## Bước 2 — Logical audit nhanh
Bước này không mở từng WAV. Nó kiểm tra manifest, transcript, split, pairing logic và số lượng tên file.

In [ ]:
LOGICAL_REPORT = REPORT_ROOT / 'logical'
logical_code = run_streaming([sys.executable, 'scripts/audit_audio_dataset.py', str(MANIFEST), '--logical-only', '--expected-clean', '8064', '--expected-noisy', '16128', '--report-dir', str(LOGICAL_REPORT)])
print('Non-zero is expected only for known V1 metadata limitations; inspect audit.json below.')


In [ ]:
import json
display(json.loads((LOGICAL_REPORT / 'audit.json').read_text(encoding='utf-8')))
recovery = DATASET_ROOT / 'manifest_recovery_report.json'
if recovery.is_file(): display(json.loads(recovery.read_text(encoding='utf-8')))


## Bước 3 — Physical audit đầy đủ (tùy chọn nhưng bắt buộc trước nghiệm thu)
Đổi `RUN_FULL_PHYSICAL_AUDIT = True` khi muốn mở và kiểm tra toàn bộ 24.192 WAV. Log xuất ngay ở file đầu tiên rồi mỗi 100 file.

In [ ]:
RUN_FULL_PHYSICAL_AUDIT = False
PHYSICAL_REPORT = REPORT_ROOT / 'physical'
if RUN_FULL_PHYSICAL_AUDIT:
    physical_code = run_streaming([sys.executable, 'scripts/audit_audio_dataset.py', str(MANIFEST), '--expected-clean', '8064', '--expected-noisy', '16128', '--workers', '8', '--progress-every', '100', '--report-dir', str(PHYSICAL_REPORT)])
    print('Physical audit exit code:', physical_code)
    display(json.loads((PHYSICAL_REPORT / 'audit.json').read_text(encoding='utf-8')))
else:
    print('Skipped full physical audit. Set RUN_FULL_PHYSICAL_AUDIT=True when ready for the long Drive scan.')
